# 🚀 Titan-GPT V3 Training on Google Colab (Fixed)

This notebook trains the Titan-GPT V3 model with the **batch size synchronization fix**.

## Features:
- ✅ **FIXED**: Batch size mismatch error in memory buffers
- ✅ GPU acceleration (T4/V100/A100)
- ✅ Automatic Google Drive mounting
- ✅ Checkpoint saving to Drive
- ✅ Progress monitoring
- ✅ Result visualization

## Setup Requirements:
1. Enable GPU: Runtime → Change runtime type → GPU
2. Run all cells in order
3. Authorize Google Drive access when prompted

## 📋 Step 1: Check GPU Availability

In [ ]:
import torch
import os

# Check GPU
if torch.cuda.is_available():
    print("✅ GPU is available!")
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ GPU not available! Please enable GPU:")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU")

## 💾 Step 2: Mount Google Drive

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create directory for checkpoints
checkpoint_dir = '/content/drive/MyDrive/Titan_V3_Checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"📁 Checkpoint directory: {checkpoint_dir}")

## 📦 Step 3: Install Dependencies

In [ ]:
!pip install -q tiktoken matplotlib tqdm
print("✅ Dependencies installed")

## 📥 Step 4: Clone Repository

In [ ]:
import sys

# Clone repository if not exists
if not os.path.exists('/content/LLMs_from_scratch'):
    print("📥 Cloning repository...")
    !git clone --depth 1 https://github.com/rasbt/LLMs-from-scratch.git /content/LLMs_from_scratch
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

# Add to path
sys.path.insert(0, '/content/LLMs_from_scratch')
sys.path.insert(0, '/content/LLMs_from_scratch/ch04/01_main_chapter_code')
sys.path.insert(0, '/content/LLMs_from_scratch/titan_optimal')

print("✅ Python paths configured")

## 🔧 Step 5: Apply Batch Size Fix

This cell applies the critical batch size synchronization fix to prevent the RuntimeError.

In [ ]:
# Read and patch neural_memory_v3.py
memory_file = '/content/LLMs_from_scratch/titan_optimal/models/neural_memory_v3.py'

with open(memory_file, 'r') as f:
    content = f.read()

# Check if fix is already applied
if 'def resize_batch(self, new_batch_size: int):' not in content:
    print("📝 Applying batch size fix to neural_memory_v3.py...")
    
    # Add resize_batch to HierarchicalMemoryV3
    hierarchical_fix = '''
    
    def resize_batch(self, new_batch_size: int):
        """Resize all memory buffers for new batch size."""
        if new_batch_size == self.batch_size:
            return
        
        device = self.medium_keys.device
        self.short_term_buffer.resize_batch(new_batch_size)
        
        if new_batch_size > self.batch_size:
            extra_batches = new_batch_size - self.batch_size
            self.medium_keys = torch.cat([self.medium_keys, torch.zeros(extra_batches, self.medium_term_size, self.dim, device=device)], dim=0)
            self.medium_values = torch.cat([self.medium_values, torch.zeros(extra_batches, self.medium_term_size, self.dim, device=device)], dim=0)
            self.medium_importance = torch.cat([self.medium_importance, torch.zeros(extra_batches, self.medium_term_size, device=device)], dim=0)
            self.medium_timestamps = torch.cat([self.medium_timestamps, torch.zeros(extra_batches, self.medium_term_size, device=device)], dim=0)
            
            if self.use_compression:
                compressed_dim = self.dim // self.compression.compression_ratio
                self.long_keys = torch.cat([self.long_keys, torch.zeros(extra_batches, self.long_term_size, compressed_dim, device=device)], dim=0)
                self.long_values = torch.cat([self.long_values, torch.zeros(extra_batches, self.long_term_size, compressed_dim, device=device)], dim=0)
            else:
                self.long_keys = torch.cat([self.long_keys, torch.zeros(extra_batches, self.long_term_size, self.dim, device=device)], dim=0)
                self.long_values = torch.cat([self.long_values, torch.zeros(extra_batches, self.long_term_size, self.dim, device=device)], dim=0)
            
            self.long_importance = torch.cat([self.long_importance, torch.zeros(extra_batches, self.long_term_size, device=device)], dim=0)
            self.long_timestamps = torch.cat([self.long_timestamps, torch.zeros(extra_batches, self.long_term_size, device=device)], dim=0)
            self.long_access_count = torch.cat([self.long_access_count, torch.zeros(extra_batches, self.long_term_size, device=device)], dim=0)
        else:
            self.medium_keys = self.medium_keys[:new_batch_size]
            self.medium_values = self.medium_values[:new_batch_size]
            self.medium_importance = self.medium_importance[:new_batch_size]
            self.medium_timestamps = self.medium_timestamps[:new_batch_size]
            self.long_keys = self.long_keys[:new_batch_size]
            self.long_values = self.long_values[:new_batch_size]
            self.long_importance = self.long_importance[:new_batch_size]
            self.long_timestamps = self.long_timestamps[:new_batch_size]
            self.long_access_count = self.long_access_count[:new_batch_size]
        
        self.batch_size = new_batch_size
'''
    
    # Find and replace
    content = content.replace(
        '        self.global_timestamp = torch.tensor(0)',
        '        self.global_timestamp = torch.tensor(0)' + hierarchical_fix
    )
    
    # Add resize_batch to NeuralMemoryV3
    neural_memory_fix = '''
    
    def resize_batch(self, new_batch_size: int):
        """Resize memory buffers for new batch size."""
        if new_batch_size != self.batch_size:
            self.hierarchical_memory.resize_batch(new_batch_size)
            self.batch_size = new_batch_size
'''
    
    content = content.replace(
        '            self.boundary_detector.reset()',
        '            self.boundary_detector.reset()' + neural_memory_fix
    )
    
    with open(memory_file, 'w') as f:
        f.write(content)
    
    print("✅ Batch size fix applied to neural_memory_v3.py")
else:
    print("✅ Batch size fix already applied")

# Patch titan_gpt_v3.py
gpt_file = '/content/LLMs_from_scratch/titan_optimal/models/titan_gpt_v3.py'

with open(gpt_file, 'r') as f:
    content = f.read()

if 'def resize_batch(self, new_batch_size: int):' not in content:
    print("📝 Applying batch size fix to titan_gpt_v3.py...")
    
    # Add resize_batch to TitanTransformerBlockV3
    block_fix = '''
    
    def resize_batch(self, new_batch_size: int):
        """Resize memory buffers for new batch size."""
        if self.use_memory:
            self.neural_memory.resize_batch(new_batch_size)
'''
    
    content = content.replace(
        '        return x, aux_losses',
        '        return x, aux_losses' + block_fix,
        1  # Only first occurrence
    )
    
    # Update forward to call resize_batch
    content = content.replace(
        '        # Update batch size if changed\n        if batch_size != self.batch_size:\n            self.batch_size = batch_size',
        '        # Update batch size if changed\n        if batch_size != self.batch_size:\n            self.batch_size = batch_size\n            # Resize memory buffers in all blocks\n            for block in self.trf_blocks:\n                block.resize_batch(batch_size)'
    )
    
    with open(gpt_file, 'w') as f:
        f.write(content)
    
    print("✅ Batch size fix applied to titan_gpt_v3.py")
else:
    print("✅ Batch size fix already applied")

print("\n✅ All batch size fixes applied successfully!")

## 📚 Step 6: Load Training Data

In [ ]:
import requests

# Download training data
data_path = '/content/the_verdict.txt'

if not os.path.exists(data_path):
    print("📥 Downloading training data...")
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main_chapter_code/the-verdict.txt"
    response = requests.get(url, timeout=30)
    with open(data_path, "w", encoding="utf-8") as f:
        f.write(response.text)
    print("✅ Training data downloaded")
else:
    print("✅ Training data already exists")

with open(data_path, "r", encoding="utf-8") as f:
    text_data = f.read()

print(f"\n📊 Loaded {len(text_data):,} characters")
print(f"📊 Preview: {text_data[:200]}...")

## 🏗️ Step 7: Create Model and Training Setup

In [ ]:
import sys
sys.path.insert(0, '/content/LLMs_from_scratch/titan_optimal')

from models.titan_gpt_v3 import TitanGPTModelV3
from configs.model_configs_v3 import get_model_config_v3, get_training_config_v3
from gpt import create_dataloader_v1
import torch.nn as nn

# Configuration
model_size = "small"  # or "medium" for larger model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*80)
print("TITAN-GPT V3 TRAINING SETUP")
print("="*80)
print(f"Device: {device}")
print(f"Model size: {model_size}")

# Get configurations
train_cfg = get_training_config_v3(model_size)
model_cfg = get_model_config_v3(model_size, use_memory=True)
model_cfg["batch_size"] = train_cfg["batch_size"]

print(f"\nModel configuration:")
print(f"  Embedding dim: {model_cfg['emb_dim']}")
print(f"  Layers: {model_cfg['n_layers']}")
print(f"  Heads: {model_cfg['n_heads']}")
print(f"  Context length: {model_cfg['context_length']}")
print(f"  Batch size: {model_cfg['batch_size']}")

# Create dataloaders
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
context_length = model_cfg['context_length']

train_loader = create_dataloader_v1(
    text_data[:split_idx],
    batch_size=train_cfg["batch_size"],
    max_length=context_length,
    stride=context_length,
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    text_data[split_idx:],
    batch_size=train_cfg["batch_size"],
    max_length=context_length,
    stride=context_length,
    drop_last=False,
    shuffle=False,
    num_workers=0
)

print(f"\nDataloaders:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

# Create model
print(f"\n{'='*80}")
print("Creating Titan-GPT V3 Model...")
print(f"{'='*80}")

model = TitanGPTModelV3(model_cfg).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model created successfully!")
print(f"   Total parameters: {total_params:,}")

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg["learning_rate"],
    weight_decay=train_cfg["weight_decay"]
)

print(f"✅ Optimizer created")
print(f"   Learning rate: {train_cfg['learning_rate']}")
print(f"   Weight decay: {train_cfg['weight_decay']}")

## 🏃 Step 8: Training Loop

In [ ]:
from tqdm.notebook import tqdm
import time
import json

# Training configuration
num_epochs = train_cfg["num_epochs"]
eval_freq = train_cfg["eval_freq"]
eval_iter = train_cfg["eval_iter"]
gradient_accumulation_steps = train_cfg["gradient_accumulation_steps"]
diversity_weight = train_cfg["diversity_loss_weight"]

# Tracking
train_losses = []
val_losses = []
perplexities = []
tokens_seen = 0
global_step = 0
best_val_loss = float('inf')

print("="*80)
print("STARTING TRAINING")
print("="*80)
print(f"Epochs: {num_epochs}")
print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
print(f"Diversity weight: {diversity_weight}")
print("="*80)

model.train()

for epoch in range(num_epochs):
    print(f"\n{'='*80}")
    print(f"EPOCH {epoch+1}/{num_epochs}")
    print(f"{'='*80}")
    
    epoch_start = time.time()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch_idx, (input_batch, target_batch) in enumerate(pbar):
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)
        
        # Forward pass
        logits, aux_losses = model(
            input_batch,
            targets=target_batch,
            update_memory=True,
            mode="train"
        )
        
        # Main loss
        main_loss = nn.functional.cross_entropy(
            logits.flatten(0, 1),
            target_batch.flatten()
        )
        
        # Add diversity loss
        diversity_loss = sum(v for k, v in aux_losses.items() if 'diversity' in k and isinstance(v, torch.Tensor))
        if isinstance(diversity_loss, torch.Tensor) and diversity_loss.numel() > 0:
            total_loss = main_loss + diversity_weight * diversity_loss
        else:
            total_loss = main_loss
        
        # Backward
        total_loss = total_loss / gradient_accumulation_steps
        total_loss.backward()
        
        tokens_seen += input_batch.numel()
        
        # Update weights
        if (batch_idx + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            
            pbar.set_postfix({'loss': f'{main_loss.item():.4f}'})
            
            # Evaluation
            if global_step % eval_freq == 0:
                model.eval()
                
                # Validation loss
                val_loss = 0.0
                with torch.no_grad():
                    for i, (val_input, val_target) in enumerate(val_loader):
                        if i >= eval_iter:
                            break
                        
                        val_input = val_input.to(device)
                        val_target = val_target.to(device)
                        
                        val_logits, _ = model(
                            val_input,
                            update_memory=False,
                            mode="inference"
                        )
                        
                        val_loss += nn.functional.cross_entropy(
                            val_logits.flatten(0, 1),
                            val_target.flatten()
                        ).item()
                
                val_loss /= min(eval_iter, len(val_loader))
                perplexity = torch.exp(torch.tensor(val_loss)).item()
                
                train_losses.append(main_loss.item())
                val_losses.append(val_loss)
                perplexities.append(perplexity)
                
                print(f"\n  Step {global_step}: Train Loss = {main_loss.item():.4f}, Val Loss = {val_loss:.4f}, PPL = {perplexity:.2f}")
                
                # Save best model
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    checkpoint_path = f"{checkpoint_dir}/titan_v3_{model_size}_best.pth"
                    torch.save({
                        'epoch': epoch + 1,
                        'global_step': global_step,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_loss': val_loss,
                        'perplexity': perplexity,
                    }, checkpoint_path)
                    print(f"  💾 Best model saved: {checkpoint_path}")
                
                model.train()
    
    epoch_time = time.time() - epoch_start
    print(f"\nEpoch {epoch+1} completed in {epoch_time:.2f}s")
    
    # Save epoch checkpoint
    checkpoint_path = f"{checkpoint_dir}/titan_v3_{model_size}_epoch_{epoch+1}.pth"
    torch.save({
        'epoch': epoch + 1,
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'perplexities': perplexities,
    }, checkpoint_path)
    print(f"💾 Epoch checkpoint saved: {checkpoint_path}")
    
    # Reset short-term memory
    model.reset_memory(level="short")

print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Final perplexity: {perplexities[-1]:.2f}")
print(f"Total tokens seen: {tokens_seen:,}")

## 📊 Step 9: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Create plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training Loss
axes[0].plot(train_losses, marker='o', label='Train Loss', markersize=3)
axes[0].set_xlabel('Evaluation Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation Loss
axes[1].plot(val_losses, marker='o', color='orange', label='Val Loss', markersize=3)
axes[1].set_xlabel('Evaluation Step')
axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Perplexity
axes[2].plot(perplexities, marker='o', color='green', label='Perplexity', markersize=3)
axes[2].set_xlabel('Evaluation Step')
axes[2].set_ylabel('Perplexity')
axes[2].set_title('Perplexity (Lower is Better)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
plot_path = f"{checkpoint_dir}/training_results.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ Plot saved to: {plot_path}")

plt.show()

# Save results to JSON
results = {
    'model_size': model_size,
    'total_params': total_params,
    'num_epochs': num_epochs,
    'best_val_loss': best_val_loss,
    'final_perplexity': perplexities[-1],
    'train_losses': train_losses,
    'val_losses': val_losses,
    'perplexities': perplexities,
}

results_path = f"{checkpoint_dir}/training_results.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved to: {results_path}")
print(f"\n📁 All files saved to: {checkpoint_dir}")

## 🎉 Training Complete!

Your model has been trained successfully with the batch size fix applied.

### Files saved to Google Drive:
- `titan_v3_{model_size}_best.pth` - Best model checkpoint
- `titan_v3_{model_size}_epoch_{n}.pth` - Epoch checkpoints
- `training_results.png` - Training visualization
- `training_results.json` - Numerical results

### Next Steps:
1. Download checkpoints from Google Drive
2. Use the model for inference
3. Fine-tune on your own data
4. Compare with baseline models